<a href="https://colab.research.google.com/github/AneelGaddam8999/Data-Driven-Supply-Chain-Optimization-and-Inventory-Analytics-Using-the-DataCo-Dataset/blob/main/notebooks/02_SQL_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/SupplyChainProject/Cleaned_DataCo.csv'
df_clean = pd.read_csv(file_path)
conn = sqlite3.connect('supply_chain_kpis.db')

df_clean.to_sql('orders', conn, if_exists='replace', index=False)
print("Database 'supply_chain_kpis.db' created and 'orders' table loaded successfully!")

test_query = """
SELECT
    category_name,
    ROUND(SUM(sales), 2) as total_sales
FROM orders
GROUP BY category_name
ORDER BY total_sales DESC
LIMIT 5;
"""
top_categories = pd.read_sql_query(test_query, conn)
print("\n--- Top 5 Product Categories by Sales ---")
display(top_categories)

Mounted at /content/drive
Database 'supply_chain_kpis.db' created and 'orders' table loaded successfully!

--- Top 5 Product Categories by Sales ---


,category_name,total_sales
0,Fishing,6929653.69
1,Cleats,4431942.78
2,Camping & Hiking,4118425.57
3,Cardio Equipment,3694843.20
4,Women's Apparel,3147800.00


In [2]:
# Query 1: Delivery Risk Analysis by Shipping Mode
# Which shipping method causes the highest percentage of late deliveries?
delivery_risk_query = """
SELECT
    shipping_mode,
    COUNT(order_id) as total_orders,
    SUM(is_late) as late_deliveries,
    ROUND((SUM(is_late) * 100.0) / COUNT(order_id), 2) as late_percentage
FROM orders
GROUP BY shipping_mode
ORDER BY late_percentage DESC;
"""
# Query 2: Profitability by Global Market
# Which markets generate the most profit and what is their average margin?
profit_market_query = """
SELECT
    market,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(benefit_per_order), 2) as total_profit,
    ROUND(AVG(profit_margin_pct), 2) as avg_profit_margin
FROM orders
GROUP BY market
ORDER BY total_profit DESC;
"""
# Query 3: Top 5 Customer Cities with the Highest Delivery Risks
# Where are the most delayed orders happening?
city_risk_query = """
SELECT
    customer_city,
    COUNT(order_id) as total_orders,
    SUM(is_late) as delayed_orders
FROM orders
GROUP BY customer_city
ORDER BY delayed_orders DESC
LIMIT 5;
"""
# Execute and display the results
print("--- Delivery Risk by Shipping Mode ---")
display(pd.read_sql_query(delivery_risk_query, conn))

print("\n--- Profitability by Market ---")
display(pd.read_sql_query(profit_market_query, conn))

print("\n--- Top 5 Riskiest Cities for Delivery ---")
display(pd.read_sql_query(city_risk_query, conn))

--- Delivery Risk by Shipping Mode ---


,shipping_mode,total_orders,late_deliveries,late_percentage
0,First Class,27814,26513,95.32
1,Second Class,35216,26987,76.63
2,Same Day,9737,4454,45.74
3,Standard Class,107752,41023,38.07



--- Profitability by Market ---


,market,total_sales,total_profit,avg_profit_margin
0,Europe,10872396.80,1169442.96,12.22
1,LATAM,10277612.84,1123321.61,12.10
2,Pacific Asia,8273743.74,857753.44,11.57
3,USCA,5066528.71,564313.78,12.16
4,Africa,2294452.93,252071.18,12.48



--- Top 5 Riskiest Cities for Delivery ---


,customer_city,total_orders,delayed_orders
0,Caguas,66770,36550
1,Chicago,3885,2198
2,Los Angeles,3417,1858
3,Brooklyn,3412,1812
4,New York,1816,1003
